# AR model diagnostics — log(Q) with monthly-mean seasonality removal

Pipeline (Salas 1993, *Handbook of Hydrology* ch. 19):

1. Monthly means
2. Log-transform  ← first, because Q is log-normal and seasonality is multiplicative
3. Detrend (OLS linear trend on log(Q))
4. Remove seasonality by subtracting each calendar month's mean
5. ACF & PACF of residual series
6. Select AR order (AIC), fit AR model
7. Compare data ACF vs model theoretical ACF
8. Residual ACF + normality test

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import (
    load_q_diepoldsau, load_q_gisingen,
    compute_monthly_mean, detrend_series,
    compute_acf_pacf, plot_acf_pacf,
    select_ar_order, fit_ar,
    plot_acf_comparison,
    plot_residual_acf, normality_test
)

## 1 — Load, monthly means, log-transform

In [ ]:
q_dp = compute_monthly_mean(load_q_diepoldsau())["Q"].dropna()
q_gi = compute_monthly_mean(load_q_gisingen())["Q"].dropna()

log_dp = np.log(q_dp)
log_gi = np.log(q_gi)
log_dp.name = "log(Q) Diepoldsau"
log_gi.name = "log(Q) Gisingen"

print(f"Diepoldsau: {len(log_dp)} months")
print(f"Gisingen:   {len(log_gi)} months")

## 2 — Detrend log(Q)

In [ ]:
det_dp, trend_dp = detrend_series(log_dp)
det_gi, trend_gi = detrend_series(log_gi)

print(f"Diepoldsau trend significant: {trend_dp['significant']}  (p={trend_dp['p_value']:.4f})")
print(f"Gisingen   trend significant: {trend_gi['significant']}  (p={trend_gi['p_value']:.4f})")

## 3 — Remove seasonality by subtracting monthly means

In [ ]:
def remove_monthly_means(series):
    monthly_means = series.groupby(series.index.month).mean()
    seasonal = series.index.month.map(monthly_means)
    return series - seasonal, monthly_means

deseas_dp, mmeans_dp = remove_monthly_means(det_dp)
deseas_gi, mmeans_gi = remove_monthly_means(det_gi)

# Plot seasonal cycle
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, mm, label in [
    (axes[0], mmeans_dp, "Diepoldsau"),
    (axes[1], mmeans_gi, "Gisingen"),
]:
    ax.bar(mm.index, mm.values, color="steelblue", alpha=0.8)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])
    ax.set_title(f"Monthly mean seasonal cycle — {label}")
    ax.set_ylabel("mean log(Q)")
plt.tight_layout()
plt.show()

## 4 — ACF & PACF of deseasonalized series

In [ ]:
corr_dp = compute_acf_pacf(deseas_dp, lags=48)
corr_gi = compute_acf_pacf(deseas_gi, lags=48)

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
plot_acf_pacf(corr_dp, title="log(Q) Diepoldsau  [deseasonalized]", axes=axes[0])
plot_acf_pacf(corr_gi, title="log(Q) Gisingen  [deseasonalized]",   axes=axes[1])
plt.tight_layout()
plt.show()

## 5 — AR order selection (AIC) & model fit

In [ ]:
p_dp, aics_dp = select_ar_order(deseas_dp, max_p=4)
p_gi, aics_gi = select_ar_order(deseas_gi, max_p=4)

print(f"Diepoldsau  best AR order: {p_dp}  |  AICs: {aics_dp}")
print(f"Gisingen    best AR order: {p_gi}  |  AICs: {aics_gi}")

ar_dp = fit_ar(deseas_dp, p_dp)
ar_gi = fit_ar(deseas_gi, p_gi)

print(f"\nAR({p_dp}) Diepoldsau — params: {ar_dp.arparams.round(4)}")
print(f"AR({p_gi}) Gisingen   — params: {ar_gi.arparams.round(4)}")

## 6 — Data ACF vs model theoretical ACF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf_comparison(corr_dp, ar_dp, title=f"AR({p_dp}) log(Q) Diepoldsau", ax=axes[0])
plot_acf_comparison(corr_gi, ar_gi, title=f"AR({p_gi}) log(Q) Gisingen",   ax=axes[1])
plt.suptitle("Observed ACF vs theoretical ACF of fitted AR model", y=1.01)
plt.tight_layout()
plt.show()

## 7 — Residual ACF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_residual_acf(ar_dp, lags=48, title=f"AR({p_dp}) log(Q) Diepoldsau", ax=axes[0])
plot_residual_acf(ar_gi, lags=48, title=f"AR({p_gi}) log(Q) Gisingen",   ax=axes[1])
plt.tight_layout()
plt.show()

## 8 — Residual probability plots (normality test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
ppcc_dp, rej_dp, _ = normality_test(ar_dp, ax=axes[0], title=f"AR({p_dp}) log(Q) Diepoldsau")
ppcc_gi, rej_gi, _ = normality_test(ar_gi, ax=axes[1], title=f"AR({p_gi}) log(Q) Gisingen")
plt.suptitle("Probability Plot — AR residuals", y=1.01)
plt.tight_layout()
plt.show()

print(f"Diepoldsau PPCC={ppcc_dp:.4f}  reject normality: {rej_dp}")
print(f"Gisingen   PPCC={ppcc_gi:.4f}  reject normality: {rej_gi}")